<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/LightBGM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import yfinance as yf

# 股票代號與名稱對應字典（以部分熱門標的為例）
stock_dict = {
      # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股 (DRAM、Flash、模組、控制晶片)】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO / 磷化銦(InP) 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝製程 / 設備概念股 (含 CoWoS、先進封裝)】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 其他電腦週邊與消費電子概念】 ---
    "2353.TW": "宏碁",
    # --- 【22. MLCC (積層陶瓷電容) 概念股】 ---
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    # --- 【23. 金融股】 ---
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    # --- 【24. 食品與零售概念股】 ---
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    # --- 【25. 力積電與成熟製程 / 特殊晶圓代工概念股】 ---
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TW": "漢磊",
    "3016.TW": "嘉晶",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理"""
  # 複製防汙染
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  # 5日收盤價線性回歸斜率
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)  # 避免除以零
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  # 跳空缺口
  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  # NATR (以 14 日 ATR 為例 / Close)
  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  # 布林頻寬 ((上軌 - 下軌) / 中軌)
  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  # 5日均線乖離率 (BIAS)
  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  # 近似 VWAP 乖離率
  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  # 能量潮 OBV 斜率
  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  # 週轉率
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  # 嚴格防洩漏：特徵全部向後位移 1 期
  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟一：正在下載大盤與個股資料，並為每支股票獨立訓練模型...")

# 下載大盤資料作為相對強度基準[span_2](start_span)[span_2](end_span)
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

# 用來存放每支股票訓練好的模型與評估結果
trained_models = {}

for ticker, name in stock_dict.items():
  try:
    print(f"\n" + "=" * 50)
    print(f" 正在處理標的：{name} ({ticker}) ")
    print("=" * 50)

    df = yf.download(ticker, period="3y", progress=False)
    if df.empty or len(df) < 300:
      print(f"{name} 資料不足，跳過訓練。")
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 目標變數定義：t+1 至 t+5 的 High 是否任一交易日 >= Close(t) * 1.05[span_3](start_span)[span_3](end_span)
    future_high_max = (
        pd.concat([df["High"].shift(-i) for i in range(1, 6)], axis=1)
        .max(axis=1)
    )
    target_condition = future_high_max >= (df["Close"] * 1.05)
    df["Target"] = target_condition.astype(int)

    # 執行特徵工程[span_4](start_span)[span_4](end_span)
    df_feat, feature_cols = compute_features(df, market_df)

    # dropna 清理因 rolling 與 shift 產生的 NaN[span_5](start_span)[span_5](end_span)
    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 100:
      print(f"{name} 清理後資料不足，跳過訓練。")
      continue

    # 依時間順序切割前 80% 訓練、後 20% 測試[span_6](start_span)[span_6](end_span)
    split_idx = int(len(df_clean) * 0.8)
    train_part = df_clean.iloc[:split_idx]
    test_part = df_clean.iloc[split_idx:]

    X_train = train_part[feature_cols].copy()
    y_train = train_part["Target"].copy()
    X_test = test_part[feature_cols].copy()
    y_test = test_part["Target"].copy()

    # 極端值處理 (Winsorization 1% 至 99%) 針對該股票獨立計算分位數[span_7](start_span)[span_7](end_span)
    for col in X_train.columns:
      lower_bound = X_train[col].quantile(0.01)
      upper_bound = X_train[col].quantile(0.99)
      X_train[col] = X_train[col].clip(lower_bound, upper_bound)
      X_test[col] = X_test[col].clip(lower_bound, upper_bound)

    print(
        f"訓練集樣本數: {len(X_train)}, 測試集樣本數: {len(X_test)}, 正樣本比例:"
        f" {y_train.mean():.4f}"
    )

    # 步驟二：為當前股票建立專屬的 LightGBM 模型並訓練[span_8](start_span)[span_8](end_span)
    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=5,
        is_unbalance=True,  # 處理類別不平衡[span_9](start_span)[span_9](end_span)
        random_state=42,
        verbose=-1,
    )

    model.fit(X_train, y_train)
    trained_models[ticker] = model  # 儲存模型字典

    # 步驟三：模型評估與輸出[span_10](start_span)[span_10](end_span)
    y_pred = model.predict(X_test)

    print(f"\n--- {name} ({ticker}) 模型績效評估報告 (測試集) ---")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # 輸出前 15 名特徵重要性[span_11](start_span)[span_11](end_span)
    feature_importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)

    print(f"\n--- {name} ({ticker}) 特徵重要性前 15 名 ---")
    print(feature_importance_df.head(15).to_markdown(index=False))

  except Exception as e:
    print(f"處理 {ticker} 時發生錯誤: {e}")
    continue

print(f"\n所有獨立模型訓練完成！總共成功訓練了 {len(trained_models)} 個股票模型。")


步驟一：正在下載大盤與個股資料，並為每支股票獨立訓練模型...


/tmp/ipykernel_998/107785350.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)



 正在處理標的：廣達 (2382.TW) 


/tmp/ipykernel_998/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


訓練集樣本數: 531, 測試集樣本數: 133, 正樣本比例: 0.3503

--- 廣達 (2382.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.6420    0.6118    0.6265        85
           1     0.3654    0.3958    0.3800        48

    accuracy                         0.5338       133
   macro avg     0.5037    0.5038    0.5033       133
weighted avg     0.5422    0.5338    0.5375       133

Confusion Matrix:
[[52 33]
 [29 19]]

--- 廣達 (2382.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| BB_Bandwidth       |          333 |
| OBV_Slope          |          218 |
| NATR               |          211 |
| Close_Slope        |          165 |
| Turnover_Rate      |          152 |
| Volume_Explosion   |          149 |
| Upper_Shadow_Ratio |          135 |
| Alpha_5d           |          114 |
| BIAS_5             |          107 |
| Lower_Shadow_Ratio |          100 |
| Gap                |           95 |
| VWAP_BIAS          |         

...

訓練集樣本數: 531, 測試集樣本數: 133, 正樣本比例: 0.3522

--- 力積電 (6770.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.5385    0.1667    0.2545        42
           1     0.7083    0.9341    0.8057        91

    accuracy                         0.6917       133
   macro avg     0.6234    0.5504    0.5301       133
weighted avg     0.6547    0.6917    0.6316       133

Confusion Matrix:
[[ 7 35]
 [ 6 85]]

--- 力積電 (6770.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| OBV_Slope          |          284 |
| BB_Bandwidth       |          239 |
| NATR               |          224 |
| Gap                |          218 |
| Turnover_Rate      |          214 |
| Upper_Shadow_Ratio |          194 |
| Close_Slope        |          188 |
| Volume_Explosion   |          133 |
| Alpha_5d           |          122 |
| Lower_Shadow_Ratio |          117 |
| VWAP_BIAS          |          106 |
| BIAS_5             |       

/tmp/ipykernel_998/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


訓練集樣本數: 531, 測試集樣本數: 133, 正樣本比例: 0.3107

--- 茂矽 (2342.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.3529    0.2667    0.3038        45
           1     0.6667    0.7500    0.7059        88

    accuracy                         0.5865       133
   macro avg     0.5098    0.5083    0.5048       133
weighted avg     0.5605    0.5865    0.5698       133

Confusion Matrix:
[[12 33]
 [22 66]]

--- 茂矽 (2342.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| NATR               |          310 |
| BB_Bandwidth       |          250 |
| Gap                |          241 |
| OBV_Slope          |          229 |
| Alpha_5d           |          185 |
| Volume_Explosion   |          181 |
| Turnover_Rate      |          174 |
| Upper_Shadow_Ratio |          153 |
| Lower_Shadow_Ratio |          129 |
| BIAS_5             |          107 |
| VWAP_BIAS          |          104 |
| Close_Slope        |         

/tmp/ipykernel_998/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['3707.TW']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')
/tmp/ipykernel_998/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


漢磊 資料不足，跳過訓練。

 正在處理標的：嘉晶 (3016.TW) 
訓練集樣本數: 531, 測試集樣本數: 133, 正樣本比例: 0.4011

--- 嘉晶 (3016.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.3103    0.2368    0.2687        38
           1     0.7212    0.7895    0.7538        95

    accuracy                         0.6316       133
   macro avg     0.5157    0.5132    0.5112       133
weighted avg     0.6038    0.6316    0.6152       133

Confusion Matrix:
[[ 9 29]
 [20 75]]

--- 嘉晶 (3016.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| NATR               |          270 |
| BB_Bandwidth       |          250 |
| Turnover_Rate      |          214 |
| OBV_Slope          |          197 |
| Close_Slope        |          176 |
| Upper_Shadow_Ratio |          170 |
| Alpha_5d           |          164 |
| Gap                |          156 |
| VWAP_BIAS          |          152 |
| Volume_Explosion   |          125 |
| Lower_Shadow_Ratio |          

In [3]:
#移除均線糾結條件
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import yfinance as yf

# 股票代號與名稱對應字典 (保留完整股票資料池)
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股 (DRAM、Flash、模組、控制晶片)】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO / 磷化銦(InP) 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝製程 / 設備概念股 (含 CoWoS、先進封裝)】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 其他電腦週邊與消費電子概念】 ---
    "2353.TW": "宏碁",
    # --- 【22. MLCC (積層陶瓷電容) 概念股】 ---
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    # --- 【23. 金融股】 ---
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    # --- 【24. 食品與零售概念股】 ---
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    # --- 【25. 力積電與成熟製程 / 特殊晶圓代工概念股】 ---
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：已移除均線糾結與量能篩選，準備對股票資料池中每檔標的進行獨立模型預測..."
)
predictions = []

for ticker, name in stock_dict.items():
  try:
    # 下載每檔股票的 2 年歷史資料[span_0](start_span)[span_0](end_span)
    df = yf.download(ticker, period="2y", progress=False)
    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .rolling(window=horizon, min_periods=1)
        .max()
        .shift(-(horizon - 1))
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行四大類特徵工程與防洩漏處理[span_1](start_span)[span_1](end_span)
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 50:
      continue

    # 訓練資料使用歷史資料（排除最後一筆正在預測當下的資料）[span_2](start_span)[span_2](end_span)
    X_train = df_clean[feature_cols].iloc[:-1]
    y_train = df_clean["Target"].iloc[:-1]

    if len(y_train.unique()) < 2:
      continue

    # 極端值處理 (Winsorization 1% 至 99%)[span_3](start_span)[span_3](end_span)
    for col in X_train.columns:
      lower_bound = X_train[col].quantile(0.01)
      upper_bound = X_train[col].quantile(0.99)
      X_train[col] = X_train[col].clip(lower_bound, upper_bound)

    train_base = y_train.mean()

    # 針對該股票訓練專屬的隨機森林模型[span_4](start_span)[span_4](end_span)
    model = RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42
    )
    model.fit(X_train, y_train)

    # 取得該股票最新一筆特徵進行預測[span_5](start_span)[span_5](end_span)
    latest_features = df_clean[feature_cols].iloc[[-1]].copy()
    for col in latest_features.columns:
      lb = X_train[col].quantile(0.01)
      ub = X_train[col].quantile(0.99)
      latest_features[col] = latest_features[col].clip(lb, ub)

    if latest_features.dropna().empty:
      continue

    prob = model.predict_proba(latest_features)[0][1]
    lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0

    predictions.append({
        "股票名稱": name,
        "股票代號": ticker.split(".")[0],
        "Base y": f"{round(float(train_base) * 100, 2)}%",
        "y成立機率值": round(float(prob), 4),
        "10% Lift 值": f"{round(lift_val, 2)}x",
    })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  print("\n" + "=" * 65)
  print(" 全資料池個股獨立模型預測結果與機率值清單 ")
  print("=" * 65)
  print(
      final_output_df[[
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前無法產生預測結果。")

步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：已移除均線糾結與量能篩選，準備對股票資料池中每檔標的進行獨立模型預測...


/tmp/ipykernel_1588/1871261672.py:319: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)
/tmp/ipykernel_1588/1871261672.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/1871261672.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/1871261672.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/1871261672.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/1871261672.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.


 全資料池個股獨立模型預測結果與機率值清單 
| 股票名稱   |   股票代號 | Base y   |   y成立機率值 | 10% Lift 值   |
|:-----------|-----------:|:---------|--------------:|:--------------|
| 廣達       |       2382 | 9.67%    |        0.2281 | 2.36x         |
| 緯創       |       3231 | 15.09%   |        0.2933 | 1.94x         |
| 緯穎       |       6669 | 34.91%   |        0.3136 | 0.9x          |
| 鴻海       |       2317 | 16.98%   |        0.1926 | 1.13x         |
| 英業達     |       2356 | 13.21%   |        0.0921 | 0.7x          |
| 仁寶       |       2324 | 12.97%   |        0.1212 | 0.93x         |
| 技嘉       |       2376 | 15.8%    |        0.0827 | 0.52x         |
| 神達       |       3706 | 25.71%   |        0.2237 | 0.87x         |
| 微星       |       2377 | 8.25%    |        0.1612 | 1.95x         |
| 華碩       |       2357 | 15.57%   |        0.2794 | 1.79x         |
| 和碩       |       4938 | 5.66%    |        0.0461 | 0.82x         |
| 神基       |       3005 | 13.44%   |        0.0597 | 0.44x         |
| 信驊       |       52

In [6]:
#保留均線個股獨立模型訓練
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import yfinance as yf

# 股票代號與名稱對應字典 (完整保留股票資料池)
stock_dict = {
   # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股 (DRAM、Flash、模組、控制晶片)】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO / 磷化銦(InP) 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝製程 / 設備概念股 (含 CoWoS、先進封裝)】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 其他電腦週邊與消費電子概念】 ---
    "2353.TW": "宏碁",
    # --- 【22. MLCC (積層陶瓷電容) 概念股】 ---
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    # --- 【23. 金融股】 ---
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    # --- 【24. 食品與零售概念股】 ---
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    # --- 【25. 力積電與成熟製程 / 特殊晶圓代工概念股】 ---
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print("步驟一：正在篩選符合流動性與均線糾結條件的股票...")
qualified_tickers = []

for ticker, name in stock_dict.items():
  try:
    df = yf.download(ticker, period="2mo", progress=False)
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df["5MA"] = df["Close"].rolling(window=5).mean()
    df["10MA"] = df["Close"].rolling(window=10).mean()
    df["20MA"] = df["Close"].rolling(window=20).mean()
    df["5MA_Vol"] = df["Volume"].rolling(window=5).mean()

    ma5 = df["5MA"].iloc[-1]
    ma10 = df["10MA"].iloc[-1]
    ma20 = df["20MA"].iloc[-1]
    vol_5ma = df["5MA_Vol"].iloc[-1]

    if pd.isna(ma5) or pd.isna(ma10) or pd.isna(ma20) or pd.isna(vol_5ma):
      continue

    # 保留篩選條件：5日均量 >= 500張 且 均線糾結絕對值 < 3%[span_0](start_span)[span_0](end_span)
    if (
        vol_5ma >= 500000
        and abs((ma5 - ma20) / ma20) < 0.03
        and abs((ma10 - ma20) / ma20) < 0.03
    ):
      qualified_tickers.append((ticker, name))
  except Exception:
    pass

print(f"篩選完成，共找到 {len(qualified_tickers)} 檔符合條件的標的。")

if len(qualified_tickers) == 0:
  print("今日盤勢中暫無符合該嚴格條件的標的。")
else:
  print("步驟二：正在分別對每檔合格股票套用進階特徵工程與獨立模型訓練預測...")
  predictions = []

  for ticker, name in qualified_tickers:
    try:
      # 下載每檔股票的 2 年歷史資料[span_1](start_span)[span_1](end_span)
      df = yf.download(ticker, period="2y", progress=False)
      if df.empty or len(df) < 250:
        continue
      if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

      # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%[span_2](start_span)[span_2](end_span)
      horizon = 10
      threshold = 0.10
      future_max = (
          df["Close"]
          .shift(-1)
          .rolling(window=horizon, min_periods=1)
          .max()
          .shift(-(horizon - 1))
      )
      future_return = (future_max - df["Close"]) / df["Close"]
      df["Target"] = (future_return >= threshold).astype(int)

      # 執行四大類特徵工程與防洩漏處理[span_3](start_span)[span_3](end_span)
      df_feat, feature_cols = compute_features(df, market_df)

      df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
      if len(df_clean) < 50:
        continue

      # 訓練資料使用歷史資料（排除最後一筆正在預測當下的資料）[span_4](start_span)[span_4](end_span)
      X_train = df_clean[feature_cols].iloc[:-1]
      y_train = df_clean["Target"].iloc[:-1]

      if len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)[span_5](start_span)[span_5](end_span)
      for col in X_train.columns:
        lower_bound = X_train[col].quantile(0.01)
        upper_bound = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower_bound, upper_bound)

      train_base = y_train.mean()

      # 針對該股票訓練專屬的獨立隨機森林模型[span_6](start_span)[span_6](end_span)
      model = RandomForestClassifier(
          n_estimators=100, max_depth=5, random_state=42
      )
      model.fit(X_train, y_train)

      # 取得該股票最新一筆特徵進行預測[span_7](start_span)[span_7](end_span)
      latest_features = df_clean[feature_cols].iloc[[-1]].copy()
      for col in latest_features.columns:
        lb = X_train[col].quantile(0.01)
        ub = X_train[col].quantile(0.99)
        latest_features[col] = latest_features[col].clip(lb, ub)

      if latest_features.dropna().empty:
        continue

      prob = model.predict_proba(latest_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0

      predictions.append({
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "y成立機率值": round(float(prob), 4),
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
    except Exception:
      pass

  final_output_df = pd.DataFrame(predictions)
  if not final_output_df.empty:
    print("\n" + "=" * 65)
    print(" 依個股獨立模型預測之結果與機率值清單 (保留篩選條件) ")
    print("=" * 65)
    print(
        final_output_df[[
            "股票名稱",
            "股票代號",
            "Base y",
            "y成立機率值",
            "10% Lift 值",
        ]].to_markdown(index=False)
    )
  else:
    print("目前無法產生預測結果。")

步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：正在篩選符合流動性與均線糾結條件的股票...


/tmp/ipykernel_1588/2344333608.py:319: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)
/tmp/ipykernel_1588/2344333608.py:328: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_1588/2344333608.py:328: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_1588/2344333608.py:328: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_1588/2344333608.py:328: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_1588/2344333608.py:328: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df =

篩選完成，共找到 36 檔符合條件的標的。
步驟二：正在分別對每檔合格股票套用進階特徵工程與獨立模型訓練預測...


/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_1588/2344333608.py:368: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download


 依個股獨立模型預測之結果與機率值清單 (保留篩選條件) 
| 股票名稱   |   股票代號 | Base y   |   y成立機率值 | 10% Lift 值   |
|:-----------|-----------:|:---------|--------------:|:--------------|
| 仁寶       |       2324 | 12.97%   |        0.1212 | 0.93x         |
| 神達       |       3706 | 25.71%   |        0.2237 | 0.87x         |
| 神基       |       3005 | 13.44%   |        0.0582 | 0.43x         |
| 佳必琪     |       6197 | 23.58%   |        0.0899 | 0.38x         |
| 智邦       |       2345 | 36.32%   |        0.389  | 1.07x         |
| 合勤控     |       3704 | 18.16%   |        0.3086 | 1.7x          |
| 勤誠       |       8210 | 33.49%   |        0.4479 | 1.34x         |
| 譜瑞-KY    |       4966 | 16.04%   |        0.0691 | 0.43x         |
| 祥碩       |       5269 | 17.92%   |        0.0873 | 0.49x         |
| 嘉澤       |       3533 | 23.82%   |        0.1273 | 0.53x         |
| 優群       |       3217 | 15.57%   |        0.1381 | 0.89x         |
| 台達電     |       2308 | 32.55%   |        0.5017 | 1.54x         |
| 群電       |    